# Step 1 (From previous notebook) 
library(Seurat)
library(Matrix)

### Load data
data <- readRDS("/mnt/lscratch/users/adhal/data/scrna_target_idf_v2/scrna_target_idf_v2/data/Immune_data/CD4_naive_to_Th17.rds")

### Get counts matrix
counts <- LayerData(data, assay = "RNA", layer = "counts")
### Or for older Seurat: counts <- GetAssayData(data, slot = "counts")

### Export directory
export_dir <- "/mnt/lscratch/users/adhal/scrna_target_idf_v3/data/from_martin/"

### 1. Save counts as Matrix Market format (efficient for sparse matrices)
writeMM(counts, file.path(export_dir, "counts.mtx"))

### 2. Save gene names
write.csv(data.frame(gene = rownames(counts)), 
          file.path(export_dir, "genes.csv"), 
          row.names = FALSE, 
          quote = FALSE)

### 3. Save cell barcodes
write.csv(data.frame(barcode = colnames(counts)), 
          file.path(export_dir, "barcodes.csv"), 
          row.names = FALSE, 
          quote = FALSE)

### 4. Save metadata
write.csv(data@meta.data, 
          file.path(export_dir, "metadata.csv"), 
          row.names = TRUE,
          quote = FALSE)

### 5. Optional: Check what other assays/reductions exist
print(names(data@assays))
print(names(data@reductions))

### 6. Optional: Export dimensionality reductions if they exist
if ("pca" %in% names(data@reductions)) {
    write.csv(Embeddings(data, reduction = "pca"),
              file.path(export_dir, "pca.csv"),
              row.names = TRUE)
}

if ("umap" %in% names(data@reductions)) {
    write.csv(Embeddings(data, reduction = "umap"),
              file.path(export_dir, "umap.csv"),
              row.names = TRUE)
}

print("Export complete!")

In [ ]:
import scanpy as sc
import pandas as pd
from scipy.io import mmread
import os

def seurat_exports_to_h5ad(data_dir, output_file):
    """Convert Seurat exports to h5ad"""
    
    # Load data
    print("Loading counts...")
    counts = mmread(os.path.join(data_dir, "counts.mtx")).T.tocsr()
    
    print("Loading genes...")
    genes = pd.read_csv(os.path.join(data_dir, "genes.csv"))['gene'].values
    
    print("Loading barcodes...")
    barcodes = pd.read_csv(os.path.join(data_dir, "barcodes.csv"))['barcode'].values
    
    print("Loading metadata...")
    metadata = pd.read_csv(os.path.join(data_dir, "metadata.csv"), index_col=0)
    
    # Create AnnData
    print("Creating AnnData...")
    adata = sc.AnnData(X=counts, obs=metadata, var=pd.DataFrame(index=genes))
    
    # Add reductions if available
    for reduction in ['pca', 'umap', 'tsne']:
        filepath = os.path.join(data_dir, f"{reduction}.csv")
        if os.path.exists(filepath):
            print(f"Adding {reduction.upper()}...")
            reduction_data = pd.read_csv(filepath, index_col=0)
            adata.obsm[f'X_{reduction}'] = reduction_data.values
    
    # Save
    print(f"Saving to {output_file}...")
    adata.write_h5ad(output_file)
    
    print(f"Done! Shape: {adata.shape}")
    return adata

# Run it
data_dir = "/mnt/lscratch/users/adhal/scrna_target_idf_v3/data/from_martin/"
adata = seurat_exports_to_h5ad(
    data_dir, 
    os.path.join(data_dir, "CD4_naive_to_Th17.h5ad")
)